<a href="https://colab.research.google.com/github/altardefey/simulador-de-impacto-orcamentario/blob/main/Simulador_Impacto_Orcamentario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulador de Impacto Orçamentário | Salvador – BA

Execute as células em ordem no Google Colab. Autorize o acesso ao Google Drive. Se a pasta Dados_turismo estiver ausente, envie o ZIP quando solicitado. Os valores gerados são estimativas condicionadas às entradas do cenário. A diferença de ISS exibida é uma comparação com o histórico, sem interpretação causal.

In [36]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile

import pandas as pd

pasta_dados = Path('/content/drive/MyDrive/Dados_turismo')
if not pasta_dados.is_dir():
    from google.colab import drive

    drive.mount('/content/drive')

colunas_por_base = [
    {'taxa_ocupacao'},
    {'diaria_media', 'revpar'},
    {'fator_evento'},
    {'receita_turistica'},
    {'receita_iss'},
]

def ler_csvs():
    if pasta_dados.is_dir():
        for arquivo in sorted(pasta_dados.glob('*.csv')):
            yield pd.read_csv(arquivo, encoding='utf-8-sig')
        return

    from google.colab import files

    print(f'Pasta ausente: {pasta_dados}. Envie o arquivo big data.zip.')
    uploads = files.upload()
    zips = [conteudo for nome, conteudo in uploads.items() if nome.lower().endswith('.zip')]
    if len(zips) != 1:
        raise ValueError('Envie um unico ZIP com os cinco CSVs.')
    with ZipFile(BytesIO(zips[0])) as arquivo_zip:
        for item in arquivo_zip.infolist():
            if item.filename.lower().endswith('.csv'):
                with arquivo_zip.open(item) as csv:
                    yield pd.read_csv(csv, encoding='utf-8-sig')

bases = {}
for tabela in ler_csvs():
    for colunas in colunas_por_base:
        if colunas <= set(tabela.columns):
            chave = frozenset(colunas)
            if chave in bases:
                raise ValueError(f'Mais de um CSV para {sorted(colunas)}.')
            bases[chave] = tabela[['ano_mes', *sorted(colunas)]].set_index('ano_mes', verify_integrity=True)

faltantes = [sorted(c) for c in colunas_por_base if frozenset(c) not in bases]
if faltantes:
    raise ValueError(f'Bases ausentes: {faltantes}')

meses = next(iter(bases.values())).index
for tabela in bases.values():
    if set(tabela.index) != set(meses):
        raise ValueError(f'Cobertura mensal diferente na base {list(tabela.columns)}.')

df_final = pd.concat(bases.values(), axis=1).sort_index()
df_final.index = pd.PeriodIndex(df_final.index, freq='M', name='ano_mes')
if not df_final.index.equals(pd.period_range(df_final.index.min(), df_final.index.max(), freq='M')):
    raise ValueError('Há lacunas ou duplicatas na sequência mensal.')
if df_final.isna().any().any():
    raise ValueError('Há valores ausentes na tabela consolidada.')
df_final['mes'] = df_final.index.month
print(f'{len(df_final)} meses consolidados: {df_final.index.min()} a {df_final.index.max()}.')
display(df_final.head())

68 meses consolidados: 2021-01 a 2026-08.


,diaria_media,revpar,fator_evento,receita_iss,receita_turistica,taxa_ocupacao,mes
ano_mes,,,,,,,
2021-01,302.71,171.33,2,80272069.56,861629655.0,46.32,1
2021-02,291.01,133.98,2,86732424.40,709653958.0,38.15,2
2021-03,268.67,63.35,0,92669109.14,332597976.0,17.88,3
2021-04,264.05,62.32,0,87733324.20,373893697.0,20.10,4
2021-05,253.39,82.57,0,87708104.97,486433840.0,26.15,5


## Treino e validação

O RevPAR informado nos CSVs diverge da fórmula do painel em parte do histórico. Para usar a mesma definição no treino e na simulação, o modelo recebe o RevPAR calculado por ocupação × diária, com ocupação convertida de percentual para fração. O RevPAR original continua na tabela para comparação.

Os 12 meses mais recentes ficam reservados para teste. A referência simples repete o valor observado no mesmo mês do ano anterior. O MAE mede o erro absoluto médio em reais e o R² pode ser negativo. Os modelos são testados com ocupação e diária observadas, portanto as métricas avaliam estimativas condicionadas a essas entradas.

In [37]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

def moeda(valor):
    numero = f'{abs(valor):,.2f}'.replace(',', '_').replace('.', ',').replace('_', '.')
    sinal = '-' if valor < 0 else ''
    return f'{sinal}R$ {numero}'

def calcular_revpar(ocupacao, diaria):
    return ocupacao / 100 * diaria

df_final['revpar_calculado'] = calcular_revpar(df_final['taxa_ocupacao'], df_final['diaria_media'])
diferenca = (df_final['revpar'] - df_final['revpar_calculado']).abs()
print(f'RevPAR informado difere do calculado em mais de R$ 0,05 em {(diferenca > 0.05).sum()} meses.')

entradas = ['taxa_ocupacao', 'diaria_media', 'revpar_calculado', 'fator_evento', 'mes']
alvos = {'receita_turistica': 'Receita turística', 'receita_iss': 'ISS'}
if len(df_final) < 24:
    raise ValueError('São necessários ao menos 24 meses para este corte de validação.')

treino = df_final.iloc[:-12]
teste = df_final.iloc[-12:]
print(f'Teste: {teste.index.min()} a {teste.index.max()}.')
modelos = {}
resultados = []
for coluna, nome in alvos.items():
    modelo = RandomForestRegressor(n_estimators=300, random_state=42)
    modelo.fit(treino[entradas], treino[coluna])
    observado = teste[coluna]
    estimado = modelo.predict(teste[entradas])
    referencia = treino.loc[teste.index - 12, coluna].to_numpy()
    for metodo, valores in [('Random Forest', estimado), ('Mesmo mês do ano anterior', referencia)]:
        resultados.append({
            'Alvo': nome,
            'Método': metodo,
            'MAE (R$)': mean_absolute_error(observado, valores),
            'R²': r2_score(observado, valores),
        })
    modelos[coluna] = modelo.fit(df_final[entradas], df_final[coluna])

metricas = pd.DataFrame(resultados).round({'R²': 3})
metricas['MAE (R$)'] = metricas['MAE (R$)'].map(moeda)
display(metricas)


RevPAR informado difere do calculado em mais de R$ 0,05 em 15 meses.
Teste: 2025-09 a 2026-08.


,Alvo,Método,MAE (R$),R²
0,Receita turística,Random Forest,"R$ 146.925.124,72",0.007
1,Receita turística,Mesmo mês do ano anterior,"R$ 69.391.482,00",0.797
2,ISS,Random Forest,"R$ 35.707.720,78",-14.752
3,ISS,Mesmo mês do ano anterior,"R$ 21.910.428,96",-4.888


# Interface  (ipywidgets + Plotly)



In [38]:
# Célula 1 - Adaptador do backend: expõe simular() e metricas_modelo() no formato que a interface espera
if 'modelos' in globals():
    def simular(ocupacao, diaria, fator_evento, mes):
        revpar = calcular_revpar(ocupacao, diaria)
        cenario = pd.DataFrame([[ocupacao, diaria, revpar, fator_evento, mes]], columns=entradas)
        return {'revpar': float(revpar),
                'receita_turistica': float(modelos['receita_turistica'].predict(cenario)[0]),
                'receita_iss': float(modelos['receita_iss'].predict(cenario)[0])}

    def metricas_modelo():
        rf = {r['Alvo']: r for r in resultados if r['Método'] == 'Random Forest'}
        imp = pd.Series(modelos['receita_turistica'].feature_importances_, index=entradas)
        return {'r2_turismo': rf['Receita turística']['R²'], 'r2_iss': rf['ISS']['R²'],
                'mae_turismo': rf['Receita turística']['MAE (R$)'], 'mae_iss': rf['ISS']['MAE (R$)'],
                'importancias': imp.to_dict(),
                'tabela': [{'Alvo': r['Alvo'], 'Método': r['Método'],
                            'MAE': r['MAE (R$)'], 'R2': r['R²']} for r in resultados]}

In [39]:
# Célula 2 - Imports, configurações, estilo visual e dados da interface
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

MESES = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun',
         'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
MESES_LONGOS = ['janeiro', 'fevereiro', 'março', 'abril', 'maio', 'junho',
                'julho', 'agosto', 'setembro', 'outubro', 'novembro', 'dezembro']
ROTULOS_EVENTO = {0: 'Mês regular', 1: 'Festivais', 2: 'Alta temporada e Carnaval'}
ROTULOS_CURTOS = {0: 'Regular', 1: 'Festivais', 2: 'Alta temporada'}

DESCRICAO_DATASETS = {
    'Taxa_ocupação.csv': 'taxa_ocupacao: ocupação hoteleira média mensal (%).',
    'Diaria_media_RevPar.csv': 'diaria_media (R$) e revpar (R$).',
    'Fator_evento.csv': 'fator_evento: 0 regular, 1 festivais, 2 alta temporada e Carnaval.',
    'Receita_turistica.csv': 'receita_turistica (R$): alvo primário do modelo.',
    'Receita_iss.csv': 'receita_iss (R$): alvo secundário, ISS dos setores ligados ao turismo.',
}

# Paleta: baía (fundo escuro), maré (azul de apoio), dendê (destaque), cinza-azulado (histórico)
BAIA, MARE, DENDE, CINZA = '#0F2F4F', '#2F7DBE', '#F2A33A', '#8FA3B8'

CSS = '''<style>
@import url('https://fonts.googleapis.com/css2?family=Public+Sans:wght@400;600;800&display=swap');
.ss, .ss * { box-sizing: border-box; }
.ss { font-family: 'Public Sans', system-ui, -apple-system, 'Segoe UI', sans-serif; line-height: 1.45; font-variant-numeric: tabular-nums; }
.ss-banner { background: #0F2F4F; color: #EAF3FA; border-radius: 10px; padding: 20px 24px; margin: 0 0 14px; }
.ss-banner .t { font-size: 24px; font-weight: 800; letter-spacing: -0.01em; margin: 0 0 4px; }
.ss-banner .s { font-size: 14px; color: #B9CFE3; }
.ss-chips { margin-top: 12px; display: flex; flex-wrap: wrap; gap: 8px; }
.ss-chip { font-size: 12px; padding: 3px 10px; border-radius: 999px; background: rgba(255,255,255,.14); color: #EAF3FA; }
.ss-chip.aviso { background: #F2A33A; color: #3A2400; font-weight: 600; }
.ss-hero { background: #0F2F4F; color: #EAF3FA; border-radius: 10px; padding: 20px 24px; }
.ss-hero .rot { font-size: 14px; color: #B9CFE3; }
.ss-hero .valor { font-size: 44px; font-weight: 800; color: #F2A33A; line-height: 1.1; letter-spacing: -0.02em; margin: 2px 0 4px; }
.ss-hero .sub { font-size: 14px; }
.ss-up { color: #2E9E5B; font-weight: 600; }
.ss-down { color: #E0574F; font-weight: 600; }
.ss-hero .ss-up { color: #6EDC9A; }
.ss-hero .ss-down { color: #FF9A93; }
.ss-sec { display: flex; flex-wrap: wrap; gap: 0 28px; margin-top: 16px; }
.ss-stat { flex: 1 1 240px; border-top: 2px solid rgba(128,128,128,.45); padding-top: 10px; margin-bottom: 12px; }
.ss-stat .rot { font-size: 13px; opacity: .7; }
.ss-stat .valor { font-size: 26px; font-weight: 800; }
.ss-stat .sub { font-size: 13px; }
.ss-regua { margin-top: 14px; }
.ss-trilho { position: relative; height: 8px; border-radius: 4px; background: rgba(128,128,128,.28); }
.ss-faixa { position: absolute; top: 0; bottom: 0; border-radius: 4px; background: rgba(47,125,190,.55); }
.ss-media { position: absolute; top: -4px; bottom: -4px; width: 2px; background: currentColor; opacity: .85; }
.ss-marca { position: absolute; top: 50%; width: 16px; height: 16px; margin: -8px 0 0 -8px; border-radius: 50%; background: #F2A33A; border: 2px solid #0F2F4F; }
.ss-rotulos { display: flex; justify-content: space-between; font-size: 12px; opacity: .8; margin-top: 6px; }
.ss-nota { font-size: 12px; opacity: .7; margin-top: 4px; max-width: 62ch; }
.ss-hero .ss-trilho { background: rgba(255,255,255,.2); }
.ss-hero .ss-faixa { background: rgba(255,255,255,.4); }
.ss-hero .ss-media { background: #EAF3FA; }
.ss-h { font-size: 17px; font-weight: 800; margin: 18px 0 4px; }
.ss-p { font-size: 14px; opacity: .8; max-width: 70ch; margin: 0 0 6px; }
.ss-tabela { border-collapse: collapse; width: 100%; max-width: 660px; font-size: 14px; }
.ss-tabela th { text-align: left; font-weight: 600; padding: 6px 10px; border-bottom: 2px solid rgba(128,128,128,.5); }
.ss-tabela td { padding: 6px 10px; border-bottom: 1px solid rgba(128,128,128,.25); }
.ss-tabela .num { text-align: right; }
.ss-tabela .melhor td { font-weight: 800; }
.ss-lista { font-size: 14px; max-width: 74ch; margin: 0; padding-left: 18px; }
</style>'''


def fmt_reais(v):
    # R$ mi / R$ bi no padrão brasileiro (1.234,5)
    if abs(v) >= 1e9:
        s, suf = f'{v / 1e9:,.2f}', ' bi'
    else:
        s, suf = f'{v / 1e6:,.1f}', ' mi'
    s = s.replace(',', 'X').replace('.', ',').replace('X', '.')
    return f'R$ {s}{suf}'


def fmt_brl(v):
    # R$ com 2 casas no padrão brasileiro (1.234,56)
    return 'R$ ' + f'{v:,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')


def fmt_pct(v):
    return f'{v:.1f}'.replace('.', ',') + '%'


# Tabela histórica usada pela interface. Aceita ano_mes como coluna OU como índice (PeriodIndex)
hist = df_final.copy()
if 'ano_mes' not in hist.columns:
    hist = hist.reset_index()
hist['ano_mes'] = hist['ano_mes'].astype(str)
hist['mes'] = pd.to_datetime(hist['ano_mes']).dt.month
COL_REVPAR = 'revpar_calculado' if 'revpar_calculado' in hist.columns else 'revpar'

# Sliders limitados à faixa histórica: o Random Forest não extrapola fora dela
OCUPACAO_MIN = math.floor(hist['taxa_ocupacao'].min())
OCUPACAO_MAX = math.ceil(hist['taxa_ocupacao'].max())
OCUPACAO_PADRAO = round(float(hist['taxa_ocupacao'].median()), 1)
DIARIA_MIN = math.floor(hist['diaria_media'].min())
DIARIA_MAX = math.ceil(hist['diaria_media'].max())
DIARIA_PADRAO = round(float(hist['diaria_media'].median()))
MES_PADRAO = int(hist['mes'].iloc[-1])   # mês mais recente da base

In [40]:
# Célula 3 - Camada de serviço: usa o backend do original se existir, senão um modelo PROVISÓRIO
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

_FEATS = ['taxa_ocupacao', 'diaria_media', 'fator_evento', 'mes']
_ALVOS = ['revpar', 'receita_turistica', 'receita_iss']
_prov = {a: LinearRegression().fit(hist[_FEATS], hist[a]) for a in _ALVOS}


def _simular_provisorio(ocupacao, diaria, fator_evento, mes):
    X = pd.DataFrame([[ocupacao, diaria, fator_evento, mes]], columns=_FEATS)
    return {a: max(0.0, float(m.predict(X)[0])) for a, m in _prov.items()}


def _metricas_provisorio():
    pt = _prov['receita_turistica'].predict(hist[_FEATS])
    pi = _prov['receita_iss'].predict(hist[_FEATS])
    peso = np.abs(_prov['receita_turistica'].coef_ * hist[_FEATS].std().values)
    return {'r2_turismo': r2_score(hist['receita_turistica'], pt),
            'r2_iss': r2_score(hist['receita_iss'], pi),
            'mae_turismo': mean_absolute_error(hist['receita_turistica'], pt),
            'mae_iss': mean_absolute_error(hist['receita_iss'], pi),
            'importancias': dict(zip(_FEATS, peso / peso.sum()))}


def usando_provisorio():
    return 'simular' not in globals() or 'metricas_modelo' not in globals()


def servico_simular(ocupacao, diaria, fator_evento, mes):
    if usando_provisorio():
        return _simular_provisorio(ocupacao, diaria, fator_evento, mes)
    return simular(ocupacao, diaria, fator_evento, mes)


def servico_metricas():
    return _metricas_provisorio() if usando_provisorio() else metricas_modelo()

In [41]:
# Célula 4 - Gráficos (cada função devolve uma figura Plotly com o mesmo estilo)
MI = 1e6
CFG = {'displayModeBar': False}
FONTE = 'Public Sans, system-ui, sans-serif'
GRADE = 'rgba(128,128,128,.25)'
TEXTO = '#8A94A3'


def _milhar(v):
    return f'{v:,.0f}'.replace(',', '.')


def _estilo(fig, altura):
    fig.update_layout(height=altura, showlegend=False,
                      paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)',
                      font=dict(family=FONTE, size=13, color=TEXTO),
                      margin=dict(l=10, r=10, t=40, b=10))
    fig.update_xaxes(showgrid=False, linecolor=GRADE)
    fig.update_yaxes(gridcolor=GRADE, zeroline=False)
    return fig


def grafico_cenario(mes, res):
    # Barras: o mesmo mês em cada ano da base + o cenário simulado (última barra)
    h = hist[hist['mes'] == mes].sort_values('ano_mes')
    anos = h['ano_mes'].str[:4].tolist() + ['Cenário']
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Receita turística (R$ milhões)',
                                                        'ISS (R$ milhões)'))
    for i, col in enumerate(['receita_turistica', 'receita_iss'], start=1):
        valores = (h[col] / MI).tolist() + [res[col] / MI]
        fig.add_trace(go.Bar(x=anos, y=valores,
                             marker=dict(color=[CINZA] * (len(anos) - 1) + [DENDE]),
                             text=[_milhar(v) for v in valores], textposition='outside',
                             cliponaxis=False), row=1, col=i)
        fig.update_yaxes(visible=False, range=[0, max(valores) * 1.18], row=1, col=i)
    fig.update_annotations(font=dict(size=14, color=TEXTO))
    return _estilo(fig, 360)


def serie_temporal(coluna):
    fig = go.Figure(go.Scatter(x=hist['ano_mes'], y=hist[coluna] / MI, mode='lines+markers',
                               line=dict(color=MARE, width=2.5), marker=dict(size=5)))
    fig.update_xaxes(tickvals=hist['ano_mes'][::6].tolist())
    fig.update_yaxes(title_text='R$ milhões')
    return _estilo(fig, 320)


def sazonalidade(coluna):
    m = hist.groupby('mes')[coluna].mean().reindex(range(1, 13)) / MI
    fig = go.Figure(go.Bar(x=MESES, y=m.values, marker=dict(color=MARE)))
    fig.update_yaxes(title_text='R$ milhões (média)')
    return _estilo(fig, 320)


def efeito_evento(coluna):
    fig = go.Figure()
    for k, nome in ROTULOS_CURTOS.items():
        d = hist[hist['fator_evento'] == k]
        fig.add_trace(go.Box(y=d[coluna] / MI, name=nome, marker_color=MARE,
                             boxpoints='all', jitter=0.4, pointpos=0))
    fig.update_yaxes(title_text='R$ milhões')
    return _estilo(fig, 320)


def correlacao():
    cols = ['taxa_ocupacao', 'diaria_media', COL_REVPAR, 'fator_evento',
            'receita_turistica', 'receita_iss']
    nomes = ['Ocupação', 'Diária', 'RevPAR', 'Evento', 'Receita turística', 'ISS']
    c = hist[cols].corr().round(2)
    texto = [[f'{v:.2f}'.replace('.', ',') for v in linha] for linha in c.values]
    fig = go.Figure(go.Heatmap(z=c.values, x=nomes, y=nomes, zmin=-1, zmax=1,
                               colorscale=[[0, '#E58C84'], [0.5, '#F2F4F7'], [1, '#7DB3DD']],
                               text=texto, texttemplate='%{text}',
                               textfont=dict(color='#1B2430'), showscale=False))
    fig.update_yaxes(autorange='reversed', gridcolor='rgba(0,0,0,0)')
    return _estilo(fig, 440)


NOMES_VARIAVEIS = {'taxa_ocupacao': 'Ocupação', 'diaria_media': 'Diária',
                   'revpar_calculado': 'RevPAR calculado', 'revpar': 'RevPAR',
                   'fator_evento': 'Fator de evento', 'mes': 'Mês', 't': 'Tendência'}


def grafico_importancias(imp):
    itens = sorted(imp.items(), key=lambda kv: kv[1])
    nomes = [NOMES_VARIAVEIS.get(k, k) for k, _ in itens]
    vals = [v for _, v in itens]
    cores = [CINZA] * (len(vals) - 1) + [MARE]
    fig = go.Figure(go.Bar(x=vals, y=nomes, orientation='h', marker=dict(color=cores),
                           text=[f'{v:.0%}' for v in vals], textposition='outside',
                           cliponaxis=False))
    fig.update_xaxes(visible=False, range=[0, max(vals) * 1.15])
    fig.update_yaxes(gridcolor='rgba(0,0,0,0)')
    return _estilo(fig, 300)

In [42]:
# Célula 5 - Interface (abas) - execute esta célula por último
def _h_html(titulo, texto=''):
    p = f'<div class="ss-p">{texto}</div>' if texto else ''
    return f'{CSS}<div class="ss"><div class="ss-h">{titulo}</div>{p}</div>'


def _h(titulo, texto=''):
    # para usar com display(...)
    return HTML(_h_html(titulo, texto))


def _hw(titulo, texto=''):
    # para usar dentro de VBox/HBox (precisa ser widget)
    return widgets.HTML(_h_html(titulo, texto))


def _delta(valor, ref, nome_mes):
    pct, dif = (valor / ref - 1) * 100, valor - ref
    cls, palavra = ('ss-up', 'acima') if dif >= 0 else ('ss-down', 'abaixo')
    return (f'<span class="{cls}">{fmt_reais(abs(dif))} ({fmt_pct(abs(pct))}) '
            f'{palavra} da média de {nome_mes}</span>')


def _regua(serie, simulado, nome_mes):
    lo, hi = min(serie.min(), simulado), max(serie.max(), simulado)
    pad = (hi - lo) * 0.06 or 1
    a, b = lo - pad, hi + pad
    pos = lambda x: (x - a) / (b - a) * 100
    return (f'<div class="ss-regua"><div class="ss-trilho">'
            f'<div class="ss-faixa" style="left:{pos(serie.min()):.1f}%;'
            f'width:{pos(serie.max()) - pos(serie.min()):.1f}%"></div>'
            f'<div class="ss-media" style="left:{pos(serie.mean()):.1f}%"></div>'
            f'<div class="ss-marca" style="left:{pos(simulado):.1f}%"></div></div>'
            f'<div class="ss-rotulos"><span>{fmt_reais(serie.min())}</span>'
            f'<span>{fmt_reais(serie.max())}</span></div>'
            f'<div class="ss-nota">Faixa dos valores de {nome_mes} na base. O traço marca a média '
            f'e o ponto marca o cenário.</div></div>')


# ----- Cabeçalho -----
chip_modelo = ('<span class="ss-chip aviso">Modelo provisório: backend não conectado</span>'
               if usando_provisorio() else '<span class="ss-chip">Modelo do original conectado</span>')
cabecalho = widgets.HTML(
    f'{CSS}<div class="ss"><div class="ss-banner">'
    f'<div class="t">Simulador de Impacto Orçamentário</div>'
    f'<div class="s">Salvador (BA): receita turística e ISS a partir de um cenário hoteleiro</div>'
    f'<div class="ss-chips"><span class="ss-chip">{len(hist)} meses de dados, '
    f"de {hist['ano_mes'].min()} a {hist['ano_mes'].max()}</span>{chip_modelo}</div></div></div>")

# ----- Aba Simulador: controles -----
LARG = widgets.Layout(width='280px')
ESTILO_SLIDER = {'handle_color': DENDE}
w_ocup = widgets.FloatSlider(value=OCUPACAO_PADRAO, min=OCUPACAO_MIN, max=OCUPACAO_MAX, step=0.5,
                             continuous_update=False, readout_format='.1f',
                             layout=LARG, style=ESTILO_SLIDER)
w_diaria = widgets.IntSlider(value=DIARIA_PADRAO, min=DIARIA_MIN, max=DIARIA_MAX, step=1,
                             continuous_update=False, layout=LARG, style=ESTILO_SLIDER)
w_evento = widgets.Dropdown(options=[(r, k) for k, r in ROTULOS_EVENTO.items()], value=0,
                            layout=LARG)
w_mes = widgets.Dropdown(options=[(m.capitalize(), i + 1) for i, m in enumerate(MESES_LONGOS)],
                         value=MES_PADRAO, layout=LARG)
btn_reset = widgets.Button(description='Restaurar valores', layout=widgets.Layout(width='auto'))
saida_sim = widgets.Output(layout=widgets.Layout(flex='1 1 480px', min_width='420px',
                                                 margin='0 0 0 18px'))


def _campo(titulo, dica, w):
    rotulo = widgets.HTML(f'{CSS}<div class="ss"><div style="font-weight:600;font-size:14px">{titulo}</div>'
                          f'<div style="font-size:12px;opacity:.7">{dica}</div></div>')
    return widgets.VBox([rotulo, w], layout=widgets.Layout(margin='0 0 14px'))


painel = widgets.VBox([
    _hw('Cenário', 'Ajuste os valores e veja o impacto na arrecadação.'),
    _campo('Taxa de ocupação (%)', f'Histórico de {OCUPACAO_MIN}% a {OCUPACAO_MAX}%', w_ocup),
    _campo('Diária média (R$)', f'Histórico de R$ {DIARIA_MIN} a R$ {DIARIA_MAX}', w_diaria),
    _campo('Fator de evento', 'Carnaval, festivais e alta temporada', w_evento),
    _campo('Mês de referência', 'Base da comparação com o histórico', w_mes),
    btn_reset,
], layout=widgets.Layout(width='340px', padding='4px 18px 16px', border='1px solid rgba(128,128,128,.4)'))

_resetando = False


def atualizar_sim(_=None):
    if _resetando:
        return
    ocup, diaria, fator, mes = w_ocup.value, w_diaria.value, w_evento.value, w_mes.value
    res = servico_simular(ocup, diaria, fator, mes)
    h_mes = hist[hist['mes'] == mes]
    nome_mes = MESES_LONGOS[mes - 1]
    iss_ref, rec_ref = h_mes['receita_iss'], h_mes['receita_turistica']

    hero = (f'<div class="ss-hero"><div class="rot">ISS estimado para o cenário</div>'
            f"<div class=\"valor\">{fmt_reais(res['receita_iss'])}</div>"
            f"<div class=\"sub\">{_delta(res['receita_iss'], iss_ref.mean(), nome_mes)}</div>"
            f"{_regua(iss_ref, res['receita_iss'], nome_mes)}</div>")
    secundarios = (
        '<div class="ss-sec">'
        '<div class="ss-stat"><div class="rot">Receita turística estimada</div>'
        f"<div class=\"valor\">{fmt_reais(res['receita_turistica'])}</div>"
        f"<div class=\"sub\">{_delta(res['receita_turistica'], rec_ref.mean(), nome_mes)}</div>"
        f"{_regua(rec_ref, res['receita_turistica'], nome_mes)}</div>"
        '<div class="ss-stat"><div class="rot">RevPAR do cenário</div>'
        f"<div class=\"valor\">{fmt_brl(res['revpar'])}</div>"
        '<div class="sub">Ocupação × diária, receita por quarto disponível</div></div></div>')
    rodape = ('<div class="ss-nota">Estimativas condicionadas ao cenário informado; '
              'não indicam relação de causa e efeito.</div>')
    with saida_sim:
        clear_output(wait=True)
        display(HTML(f'{CSS}<div class="ss">{hero}{secundarios}</div>'))
        display(_h(f'{nome_mes.capitalize()} em cada ano, com o cenário',
                   'As barras cinzas são os valores reais da base; a barra amarela é o cenário.'))
        grafico_cenario(mes, res).show(config=CFG)
        display(HTML(f'{CSS}<div class="ss">{rodape}</div>'))


def restaurar(_=None):
    global _resetando
    _resetando = True
    w_ocup.value, w_diaria.value = OCUPACAO_PADRAO, DIARIA_PADRAO
    w_evento.value, w_mes.value = 0, MES_PADRAO
    _resetando = False
    atualizar_sim()


for w in (w_ocup, w_diaria, w_evento, w_mes):
    w.observe(atualizar_sim, names='value')
btn_reset.on_click(restaurar)

aba_sim = widgets.HBox([painel, saida_sim],
                       layout=widgets.Layout(flex_flow='row wrap', align_items='flex-start'))

# ----- Aba Histórico -----
w_alvo = widgets.ToggleButtons(options=[('Receita turística', 'receita_turistica'), ('ISS', 'receita_iss')])
saida_hist = widgets.Output()


def atualizar_hist(_=None):
    col = w_alvo.value
    nome = 'receita turística' if col == 'receita_turistica' else 'ISS'
    with saida_hist:
        clear_output(wait=True)
        display(_h(f'Evolução mensal da {nome}' if col == 'receita_turistica' else 'Evolução mensal do ISS',
                   f"{len(hist)} meses, de {hist['ano_mes'].min()} a {hist['ano_mes'].max()}."))
        serie_temporal(col).show(config=CFG)
        display(_h('Média por mês do ano', 'Mostra a sazonalidade: quais meses costumam render mais.'))
        sazonalidade(col).show(config=CFG)
        display(_h('Efeito do fator de evento', 'Cada ponto é um mês da base, agrupado pelo tipo de evento.'))
        efeito_evento(col).show(config=CFG)
        display(_h('Correlação entre as variáveis',
                   'Valores próximos de 1 indicam que as variáveis sobem juntas. Parte da relação '
                   'com o ISS vem da tendência de crescimento ao longo dos anos.'))
        correlacao().show(config=CFG)


w_alvo.observe(atualizar_hist, names='value')
aba_hist = widgets.VBox([w_alvo, saida_hist])

# ----- Aba Sobre o modelo -----
saida_modelo = widgets.Output()


def atualizar_modelo():
    m = servico_metricas()
    linhas = m.get('tabela') or [
        {'Alvo': 'Receita turística', 'Método': 'Modelo', 'MAE': m['mae_turismo'], 'R2': m['r2_turismo']},
        {'Alvo': 'ISS', 'Método': 'Modelo', 'MAE': m['mae_iss'], 'R2': m['r2_iss']}]
    melhor = {a: min(r['MAE'] for r in linhas if r['Alvo'] == a) for a in {r['Alvo'] for r in linhas}}
    def _linha(r):
        r2 = f"{r['R2']:.2f}".replace('.', ',')
        cls = 'melhor' if r['MAE'] == melhor[r['Alvo']] else ''
        return (f'<tr class="{cls}"><td>{r["Alvo"]}</td><td>{r["Método"]}</td>'
                f'<td class="num">{fmt_reais(r["MAE"])}</td><td class="num">{r2}</td></tr>')

    corpo = ''.join(_linha(r) for r in linhas)
    tabela = ('<table class="ss-tabela"><tr><th>Alvo</th><th>Método</th>'
              '<th class="num">Erro médio (MAE)</th><th class="num">R²</th></tr>' + corpo + '</table>')
    itens = ''.join(f'<li><b>{a}</b>: {d}</li>' for a, d in DESCRICAO_DATASETS.items())
    aviso = ('<div class="ss-nota">Métricas do modelo provisório, calculadas nos próprios dados de treino.</div>'
             if usando_provisorio() else
             '<div class="ss-nota">Avaliado nos 12 meses mais recentes da base. Em negrito, o menor erro de cada alvo. '
             'O R² pode ser negativo quando o método erra mais que a média.</div>')
    with saida_modelo:
        clear_output(wait=True)
        display(_h('Desempenho no teste', 'Compara o modelo com o valor do mesmo mês do ano anterior.'))
        display(HTML(f'{CSS}<div class="ss">{tabela}{aviso}</div>'))
        display(_h('Variáveis que mais pesam na receita turística'))
        grafico_importancias(m['importancias']).show(config=CFG)
        display(_h('Bases de dados utilizadas'))
        display(HTML(f'{CSS}<div class="ss"><ul class="ss-lista">{itens}</ul></div>'))


aba_modelo = widgets.VBox([saida_modelo])

tabs = widgets.Tab(children=[aba_sim, aba_hist, aba_modelo])
for i, t in enumerate(['Simulador', 'Histórico', 'Sobre o modelo']):
    tabs.set_title(i, t)

atualizar_sim()
atualizar_hist()
atualizar_modelo()
display(widgets.VBox([cabecalho, tabs]))